In [1]:
%%configure -f
{
    "conf":{
        "spark.executor.instances": "2",
        "spark.executor.memory": "8g",
        "spark.executor.cores": "4"
    }
}

ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
1437,application_1765289937462_1425,pyspark,idle,Link,Link,None,
1446,application_1765289937462_1434,pyspark,idle,Link,Link,None,
1448,application_1765289937462_1436,pyspark,idle,Link,Link,None,
1454,application_1765289937462_1441,pyspark,idle,Link,Link,None,
1455,application_1765289937462_1442,pyspark,idle,Link,Link,None,
1457,application_1765289937462_1444,pyspark,idle,Link,Link,None,
1458,application_1765289937462_1445,pyspark,idle,Link,Link,None,


In [2]:
from pyspark.sql import SparkSession
from sedona.spark import *
from pyspark.sql.functions import col, year, to_date, sum as _sum, avg, lit, corr
from pyspark.sql.functions import regexp_extract, col, to_timestamp
from pyspark.sql.types import DoubleType
import time

spark = SparkSession.builder.appName("Query 5").getOrCreate()
sedona = SedonaContext.create(spark)

spark.catalog.clearCache() 
global_start_time = time.time()

# Φόρτωση & Καθαρισμός CRIME DATA (2020-2021 μόνο)
crimes_df = spark.read.option("header", "true").csv("s3://initial-notebook-data-bucket-dblab-905418150721/project_data/LA_Crime_Data/LA_Crime_Data_2020_2025.csv")

crimes_filtered = crimes_df.withColumn(
    "Year", 
    regexp_extract(col("DATE OCC"), r'(\d{4})', 1).cast("int")
)

# Φιλτράρισμα και Καθαρισμός
crimes_filtered = crimes_filtered \
    .filter((col("Year").isin([2020, 2021])) & (col("LAT") != 0) & (col("LON") != 0)) \
    .withColumn("crime_geom", ST_Point(col("LON").cast(DoubleType()), col("LAT").cast(DoubleType())))

# Φόρτωση CENSUS BLOCKS (GeoJSON)
# Φορτώνουμε τα πολύγωνα και κρατάμε: COMM (Περιοχή), POP20 (Πληθυσμός), geometry
census_raw = sedona.read.format("geojson").option("multiLine", "true") \
    .load("s3://initial-notebook-data-bucket-dblab-905418150721/project_data/LA_Census_Blocks_2020.geojson") \
    .selectExpr("explode(features) as features")

census_df = census_raw.select(
    col("features.properties.COMM").alias("COMM"),
    col("features.properties.POP20").alias("POP20"),
    col("features.properties.ZCTA20").alias("ZipCode"),
    col("features.geometry").alias("census_geom")
)

# SPATIAL JOIN
# Βρίσκουμε σε ποια κοινότητα (COMM) ανήκει κάθε έγκλημα.
crime_with_comm = crimes_filtered.alias("c") \
    .join(census_df.alias("b"), ST_Within("c.crime_geom", "b.census_geom")) \
    .select("c.Year", "b.COMM")
crime_with_comm.cache()

# Κάνουμε register ως Temp Views για να πάμε σε SQL
crime_with_comm.createOrReplaceTempView("crime_data")
census_df.createOrReplaceTempView("census_data")

print("=== Spatial Join Execution Plan ===")
# Χρησιμοποιούμε mode="extended" για να δούμε όλες τις λεπτομέρειες
crime_with_comm.explain(mode="extended")

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
1463,application_1765289937462_1450,pyspark,idle,Link,Link,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

=== Spatial Join Execution Plan ===
== Parsed Logical Plan ==
'Project ['c.Year, 'b.COMM]
+- Join Inner,  **org.apache.spark.sql.sedona_sql.expressions.ST_Within**
   :- SubqueryAlias c
   :  +- Project [DR_NO#42, Date Rptd#43, DATE OCC#44, TIME OCC#45, AREA#46, AREA NAME#47, Rpt Dist No#48, Part 1-2#49, Crm Cd#50, Crm Cd Desc#51, Mocodes#52, Vict Age#53, Vict Sex#54, Vict Descent#55, Premis Cd#56, Premis Desc#57, Weapon Used Cd#58, Weapon Desc#59, Status#60, Status Desc#61, Crm Cd 1#62, Crm Cd 2#63, Crm Cd 3#64, Crm Cd 4#65, ... 6 more fields]
   :     +- Filter ((Year#98 IN (2020,2021) AND NOT (cast(LAT#68 as int) = 0)) AND NOT (cast(LON#69 as int) = 0))
   :        +- Project [DR_NO#42, Date Rptd#43, DATE OCC#44, TIME OCC#45, AREA#46, AREA NAME#47, Rpt Dist No#48, Part 1-2#49, Crm Cd#50, Crm Cd Desc#51, Mocodes#52, Vict Age#53, Vict Sex#54, Vict Descent#55, Premis Cd#56, Premis Desc#57, Weapon Used Cd#58, Weapon Desc#59, Status#60, Status Desc#61, Crm Cd 1#62, Crm Cd 2#63, Crm Cd 3#

In [3]:
from pyspark.sql.functions import split, col, element_at, regexp_replace
from pyspark.sql.types import DoubleType

# Φόρτωση INCOME DATA
income_text = spark.read.text("s3://initial-notebook-data-bucket-dblab-905418150721/project_data/LA_income_2021.csv")

# Φιλτράρουμε την επικεφαλίδα
income_data = income_text.filter(~col("value").contains("Zip Code"))

# Σπάμε τη γραμμή με το ";"
income_data = income_data.withColumn("data_cols", split(col("value"), ";"))

income_df = income_data.select(
    col("data_cols")[0].alias("ZipCode"),
    element_at(col("data_cols"), -1).alias("Raw_Income")
)

# Καθαρισμός και Casting
income_df = income_df.withColumn(
    "income_val", 
    regexp_replace(col("Raw_Income"), "[$,]", "").cast(DoubleType())
)

print("Sample Income Data:")
income_df.show(5, truncate=False)

# Δημιουργία του View
income_df.createOrReplaceTempView("income_data")

# ΤΕΛΙΚΟ SQL QUERY
final_query = """
WITH Community_Stats AS (
    SELECT 
        c.COMM, 
        SUM(c.POP20) as Total_Pop,
        AVG(i.income_val) as Avg_Income
    FROM census_data c
    JOIN income_data i ON c.ZipCode = i.ZipCode
    GROUP BY c.COMM
),
Crime_Counts AS (
    SELECT 
        COMM, 
        COUNT(*) as Total_Crimes
    FROM crime_data
    GROUP BY COMM
)
SELECT 
    s.COMM,
    s.Avg_Income,
    (c.Total_Crimes / s.Total_Pop) as Crime_Rate_Per_Capita
FROM Community_Stats s
JOIN Crime_Counts c ON s.COMM = c.COMM
WHERE s.Total_Pop > 0 AND s.Avg_Income IS NOT NULL
"""

analysis_df = spark.sql(final_query)

print("Final Analysis Results:")
analysis_df.show(5)

print("\n=== Final Query Execution Plan ===")
analysis_df.explain(mode="extended")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Sample Income Data:
+-------+----------+----------+
|ZipCode|Raw_Income|income_val|
+-------+----------+----------+
|90001  |$52,806   |52806.0   |
|90002  |$46,159   |46159.0   |
|90003  |$47,733   |47733.0   |
|90004  |$54,947   |54947.0   |
|90005  |$44,913   |44913.0   |
+-------+----------+----------+
only showing top 5 rows

Final Analysis Results:
+--------------------+------------------+---------------------+
|                COMM|        Avg_Income|Crime_Rate_Per_Capita|
+--------------------+------------------+---------------------+
|Santa Monica Moun...| 152714.4776119403| 6.091246878235975E-5|
|              Encino|         108085.75|  0.08626928626928627|
|       Cheviot Hills|        106863.272|  0.06591572799332499|
|    West Los Angeles|          101937.0|  0.11895456116561644|
|       Beverly Crest|157253.82558139536|  0.05168652458466186|
+--------------------+------------------+---------------------+
only showing top 5 rows


=== Final Query Execution Plan ===
== Par

In [4]:
# Συσχέτιση (Correlation) για ΟΛΕΣ τις περιοχές
print("Correlation (All Areas):")
analysis_df.select(corr("Avg_Income", "Crime_Rate_Per_Capita")).show()

# Top 10 Περιοχές με Υψηλότερο Εισόδημα
top10_income = analysis_df.orderBy(col("Avg_Income").desc()).limit(10)
print("Correlation (Top 10 Rich Areas):")
top10_income.select(corr("Avg_Income", "Crime_Rate_Per_Capita")).show()

# Bottom 10 Περιοχές με Χαμηλότερο Εισόδημα
bottom10_income = analysis_df.orderBy(col("Avg_Income").asc()).limit(10)
print("Correlation (Bottom 10 Poor Areas):")
bottom10_income.select(corr("Avg_Income", "Crime_Rate_Per_Capita")).show()

global_end_time = time.time()
print(f"Total Execution Time: {global_end_time - global_start_time} seconds")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Correlation (All Areas):
+---------------------------------------+
|corr(Avg_Income, Crime_Rate_Per_Capita)|
+---------------------------------------+
|                    -0.2833188038095934|
+---------------------------------------+

Correlation (Top 10 Rich Areas):
+---------------------------------------+
|corr(Avg_Income, Crime_Rate_Per_Capita)|
+---------------------------------------+
|                    0.02162780394666356|
+---------------------------------------+

Correlation (Bottom 10 Poor Areas):
+---------------------------------------+
|corr(Avg_Income, Crime_Rate_Per_Capita)|
+---------------------------------------+
|                    -0.7086199332400863|
+---------------------------------------+

Total Execution Time: 70.11449241638184 seconds

In [5]:
%%configure -f
{
    "conf":{
        "spark.executor.instances": "4",
        "spark.executor.memory": "4g",
        "spark.executor.cores": "2"
    }
}

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
1464,application_1765289937462_1451,pyspark,idle,Link,Link,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
1437,application_1765289937462_1425,pyspark,idle,Link,Link,None,
1446,application_1765289937462_1434,pyspark,idle,Link,Link,None,
1448,application_1765289937462_1436,pyspark,idle,Link,Link,None,
1454,application_1765289937462_1441,pyspark,idle,Link,Link,None,
1455,application_1765289937462_1442,pyspark,idle,Link,Link,None,
1457,application_1765289937462_1444,pyspark,idle,Link,Link,None,
1458,application_1765289937462_1445,pyspark,idle,Link,Link,None,
1464,application_1765289937462_1451,pyspark,idle,Link,Link,None,✔


In [6]:
from pyspark.sql import SparkSession
from sedona.spark import *
from pyspark.sql.functions import col, year, to_date, sum as _sum, avg, lit, corr
from pyspark.sql.functions import regexp_extract, col, to_timestamp
from pyspark.sql.types import DoubleType
import time

spark = SparkSession.builder.appName("Query 5").getOrCreate()
sedona = SedonaContext.create(spark)

spark.catalog.clearCache() 
global_start_time = time.time()

# Φόρτωση & Καθαρισμός CRIME DATA (2020-2021 μόνο)
crimes_df = spark.read.option("header", "true").csv("s3://initial-notebook-data-bucket-dblab-905418150721/project_data/LA_Crime_Data/LA_Crime_Data_2020_2025.csv")

crimes_filtered = crimes_df.withColumn(
    "Year", 
    regexp_extract(col("DATE OCC"), r'(\d{4})', 1).cast("int")
)

# Φιλτράρισμα και Καθαρισμός
crimes_filtered = crimes_filtered \
    .filter((col("Year").isin([2020, 2021])) & (col("LAT") != 0) & (col("LON") != 0)) \
    .withColumn("crime_geom", ST_Point(col("LON").cast(DoubleType()), col("LAT").cast(DoubleType())))

# Φόρτωση CENSUS BLOCKS (GeoJSON)
# Φορτώνουμε τα πολύγωνα και κρατάμε: COMM (Περιοχή), POP20 (Πληθυσμός), geometry
census_raw = sedona.read.format("geojson").option("multiLine", "true") \
    .load("s3://initial-notebook-data-bucket-dblab-905418150721/project_data/LA_Census_Blocks_2020.geojson") \
    .selectExpr("explode(features) as features")

census_df = census_raw.select(
    col("features.properties.COMM").alias("COMM"),
    col("features.properties.POP20").alias("POP20"),
    col("features.properties.ZCTA20").alias("ZipCode"),
    col("features.geometry").alias("census_geom")
)

# SPATIAL JOIN
# Βρίσκουμε σε ποια κοινότητα (COMM) ανήκει κάθε έγκλημα.
crime_with_comm = crimes_filtered.alias("c") \
    .join(census_df.alias("b"), ST_Within("c.crime_geom", "b.census_geom")) \
    .select("c.Year", "b.COMM")
crime_with_comm.cache()

# Κάνουμε register ως Temp Views για να πάμε σε SQL
crime_with_comm.createOrReplaceTempView("crime_data")
census_df.createOrReplaceTempView("census_data")

print("=== Spatial Join Execution Plan ===")
# Χρησιμοποιούμε mode="extended" για να δούμε όλες τις λεπτομέρειες
crime_with_comm.explain(mode="extended")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

=== Spatial Join Execution Plan ===
== Parsed Logical Plan ==
'Project ['c.Year, 'b.COMM]
+- Join Inner,  **org.apache.spark.sql.sedona_sql.expressions.ST_Within**
   :- SubqueryAlias c
   :  +- Project [DR_NO#42, Date Rptd#43, DATE OCC#44, TIME OCC#45, AREA#46, AREA NAME#47, Rpt Dist No#48, Part 1-2#49, Crm Cd#50, Crm Cd Desc#51, Mocodes#52, Vict Age#53, Vict Sex#54, Vict Descent#55, Premis Cd#56, Premis Desc#57, Weapon Used Cd#58, Weapon Desc#59, Status#60, Status Desc#61, Crm Cd 1#62, Crm Cd 2#63, Crm Cd 3#64, Crm Cd 4#65, ... 6 more fields]
   :     +- Filter ((Year#98 IN (2020,2021) AND NOT (cast(LAT#68 as int) = 0)) AND NOT (cast(LON#69 as int) = 0))
   :        +- Project [DR_NO#42, Date Rptd#43, DATE OCC#44, TIME OCC#45, AREA#46, AREA NAME#47, Rpt Dist No#48, Part 1-2#49, Crm Cd#50, Crm Cd Desc#51, Mocodes#52, Vict Age#53, Vict Sex#54, Vict Descent#55, Premis Cd#56, Premis Desc#57, Weapon Used Cd#58, Weapon Desc#59, Status#60, Status Desc#61, Crm Cd 1#62, Crm Cd 2#63, Crm Cd 3#

In [7]:
from pyspark.sql.functions import split, col, element_at, regexp_replace
from pyspark.sql.types import DoubleType

# Φόρτωση INCOME DATA
income_text = spark.read.text("s3://initial-notebook-data-bucket-dblab-905418150721/project_data/LA_income_2021.csv")

# Φιλτράρουμε την επικεφαλίδα
income_data = income_text.filter(~col("value").contains("Zip Code"))

# Σπάμε τη γραμμή με το ";"
income_data = income_data.withColumn("data_cols", split(col("value"), ";"))

income_df = income_data.select(
    col("data_cols")[0].alias("ZipCode"),
    element_at(col("data_cols"), -1).alias("Raw_Income")
)

# Καθαρισμός και Casting
income_df = income_df.withColumn(
    "income_val", 
    regexp_replace(col("Raw_Income"), "[$,]", "").cast(DoubleType())
)

print("Sample Income Data:")
income_df.show(5, truncate=False)

# Δημιουργία του View
income_df.createOrReplaceTempView("income_data")

# ΤΕΛΙΚΟ SQL QUERY
final_query = """
WITH Community_Stats AS (
    SELECT 
        c.COMM, 
        SUM(c.POP20) as Total_Pop,
        AVG(i.income_val) as Avg_Income
    FROM census_data c
    JOIN income_data i ON c.ZipCode = i.ZipCode
    GROUP BY c.COMM
),
Crime_Counts AS (
    SELECT 
        COMM, 
        COUNT(*) as Total_Crimes
    FROM crime_data
    GROUP BY COMM
)
SELECT 
    s.COMM,
    s.Avg_Income,
    (c.Total_Crimes / s.Total_Pop) as Crime_Rate_Per_Capita
FROM Community_Stats s
JOIN Crime_Counts c ON s.COMM = c.COMM
WHERE s.Total_Pop > 0 AND s.Avg_Income IS NOT NULL
"""

analysis_df = spark.sql(final_query)

print("Final Analysis Results:")
analysis_df.show(5)

print("\n=== Final Query Execution Plan ===")
analysis_df.explain(mode="extended")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Sample Income Data:
+-------+----------+----------+
|ZipCode|Raw_Income|income_val|
+-------+----------+----------+
|90001  |$52,806   |52806.0   |
|90002  |$46,159   |46159.0   |
|90003  |$47,733   |47733.0   |
|90004  |$54,947   |54947.0   |
|90005  |$44,913   |44913.0   |
+-------+----------+----------+
only showing top 5 rows

Final Analysis Results:
+-----------------+-----------------+---------------------+
|             COMM|       Avg_Income|Crime_Rate_Per_Capita|
+-----------------+-----------------+---------------------+
| Lafayette Square| 61321.2962962963|   0.0982294445791899|
|  Wilshire Center|52797.35820895522|  0.11456896002717218|
|Country Club Park|58138.31460674157|  0.10466609589041095|
|        Koreatown| 48248.2816091954|  0.11184641732033744|
|  Harvard Heights|          48653.1|  0.10957864102239656|
+-----------------+-----------------+---------------------+
only showing top 5 rows


=== Final Query Execution Plan ===
== Parsed Logical Plan ==
CTE [Community_S

In [8]:
# Συσχέτιση (Correlation) για ΟΛΕΣ τις περιοχές
print("Correlation (All Areas):")
analysis_df.select(corr("Avg_Income", "Crime_Rate_Per_Capita")).show()

# Top 10 Περιοχές με Υψηλότερο Εισόδημα
top10_income = analysis_df.orderBy(col("Avg_Income").desc()).limit(10)
print("Correlation (Top 10 Rich Areas):")
top10_income.select(corr("Avg_Income", "Crime_Rate_Per_Capita")).show()

# Bottom 10 Περιοχές με Χαμηλότερο Εισόδημα
bottom10_income = analysis_df.orderBy(col("Avg_Income").asc()).limit(10)
print("Correlation (Bottom 10 Poor Areas):")
bottom10_income.select(corr("Avg_Income", "Crime_Rate_Per_Capita")).show()

global_end_time = time.time()
print(f"Total Execution Time: {global_end_time - global_start_time} seconds")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Correlation (All Areas):
+---------------------------------------+
|corr(Avg_Income, Crime_Rate_Per_Capita)|
+---------------------------------------+
|                   -0.28331880380959346|
+---------------------------------------+

Correlation (Top 10 Rich Areas):
+---------------------------------------+
|corr(Avg_Income, Crime_Rate_Per_Capita)|
+---------------------------------------+
|                    0.02162780394666356|
+---------------------------------------+

Correlation (Bottom 10 Poor Areas):
+---------------------------------------+
|corr(Avg_Income, Crime_Rate_Per_Capita)|
+---------------------------------------+
|                    -0.7086199332400863|
+---------------------------------------+

Total Execution Time: 90.40760946273804 seconds

In [9]:
%%configure -f
{
    "conf":{
        "spark.executor.instances": "8",
        "spark.executor.memory": "2g",
        "spark.executor.cores": "1"
    }
}

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
1465,application_1765289937462_1452,pyspark,idle,Link,Link,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
1437,application_1765289937462_1425,pyspark,idle,Link,Link,None,
1446,application_1765289937462_1434,pyspark,idle,Link,Link,None,
1448,application_1765289937462_1436,pyspark,idle,Link,Link,None,
1454,application_1765289937462_1441,pyspark,idle,Link,Link,None,
1455,application_1765289937462_1442,pyspark,idle,Link,Link,None,
1457,application_1765289937462_1444,pyspark,idle,Link,Link,None,
1458,application_1765289937462_1445,pyspark,idle,Link,Link,None,
1465,application_1765289937462_1452,pyspark,idle,Link,Link,None,✔


In [10]:
from pyspark.sql import SparkSession
from sedona.spark import *
from pyspark.sql.functions import col, year, to_date, sum as _sum, avg, lit, corr
from pyspark.sql.functions import regexp_extract, col, to_timestamp
from pyspark.sql.types import DoubleType
import time

spark = SparkSession.builder.appName("Query 5").getOrCreate()
sedona = SedonaContext.create(spark)

spark.catalog.clearCache() 
global_start_time = time.time()

# Φόρτωση & Καθαρισμός CRIME DATA (2020-2021 μόνο)
crimes_df = spark.read.option("header", "true").csv("s3://initial-notebook-data-bucket-dblab-905418150721/project_data/LA_Crime_Data/LA_Crime_Data_2020_2025.csv")

crimes_filtered = crimes_df.withColumn(
    "Year", 
    regexp_extract(col("DATE OCC"), r'(\d{4})', 1).cast("int")
)

# Φιλτράρισμα και Καθαρισμός
crimes_filtered = crimes_filtered \
    .filter((col("Year").isin([2020, 2021])) & (col("LAT") != 0) & (col("LON") != 0)) \
    .withColumn("crime_geom", ST_Point(col("LON").cast(DoubleType()), col("LAT").cast(DoubleType())))

# Φόρτωση CENSUS BLOCKS (GeoJSON)
# Φορτώνουμε τα πολύγωνα και κρατάμε: COMM (Περιοχή), POP20 (Πληθυσμός), geometry
census_raw = sedona.read.format("geojson").option("multiLine", "true") \
    .load("s3://initial-notebook-data-bucket-dblab-905418150721/project_data/LA_Census_Blocks_2020.geojson") \
    .selectExpr("explode(features) as features")

census_df = census_raw.select(
    col("features.properties.COMM").alias("COMM"),
    col("features.properties.POP20").alias("POP20"),
    col("features.properties.ZCTA20").alias("ZipCode"),
    col("features.geometry").alias("census_geom")
)

# SPATIAL JOIN
# Βρίσκουμε σε ποια κοινότητα (COMM) ανήκει κάθε έγκλημα.
crime_with_comm = crimes_filtered.alias("c") \
    .join(census_df.alias("b"), ST_Within("c.crime_geom", "b.census_geom")) \
    .select("c.Year", "b.COMM")
crime_with_comm.cache()

# Κάνουμε register ως Temp Views για να πάμε σε SQL
crime_with_comm.createOrReplaceTempView("crime_data")
census_df.createOrReplaceTempView("census_data")

print("=== Spatial Join Execution Plan ===")
# Χρησιμοποιούμε mode="extended" για να δούμε όλες τις λεπτομέρειες
crime_with_comm.explain(mode="extended")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

=== Spatial Join Execution Plan ===
== Parsed Logical Plan ==
'Project ['c.Year, 'b.COMM]
+- Join Inner,  **org.apache.spark.sql.sedona_sql.expressions.ST_Within**
   :- SubqueryAlias c
   :  +- Project [DR_NO#42, Date Rptd#43, DATE OCC#44, TIME OCC#45, AREA#46, AREA NAME#47, Rpt Dist No#48, Part 1-2#49, Crm Cd#50, Crm Cd Desc#51, Mocodes#52, Vict Age#53, Vict Sex#54, Vict Descent#55, Premis Cd#56, Premis Desc#57, Weapon Used Cd#58, Weapon Desc#59, Status#60, Status Desc#61, Crm Cd 1#62, Crm Cd 2#63, Crm Cd 3#64, Crm Cd 4#65, ... 6 more fields]
   :     +- Filter ((Year#98 IN (2020,2021) AND NOT (cast(LAT#68 as int) = 0)) AND NOT (cast(LON#69 as int) = 0))
   :        +- Project [DR_NO#42, Date Rptd#43, DATE OCC#44, TIME OCC#45, AREA#46, AREA NAME#47, Rpt Dist No#48, Part 1-2#49, Crm Cd#50, Crm Cd Desc#51, Mocodes#52, Vict Age#53, Vict Sex#54, Vict Descent#55, Premis Cd#56, Premis Desc#57, Weapon Used Cd#58, Weapon Desc#59, Status#60, Status Desc#61, Crm Cd 1#62, Crm Cd 2#63, Crm Cd 3#

In [11]:
from pyspark.sql.functions import split, col, element_at, regexp_replace
from pyspark.sql.types import DoubleType

# Φόρτωση INCOME DATA
income_text = spark.read.text("s3://initial-notebook-data-bucket-dblab-905418150721/project_data/LA_income_2021.csv")

# Φιλτράρουμε την επικεφαλίδα
income_data = income_text.filter(~col("value").contains("Zip Code"))

# Σπάμε τη γραμμή με το ";"
income_data = income_data.withColumn("data_cols", split(col("value"), ";"))

income_df = income_data.select(
    col("data_cols")[0].alias("ZipCode"),
    element_at(col("data_cols"), -1).alias("Raw_Income")
)

# Καθαρισμός και Casting
income_df = income_df.withColumn(
    "income_val", 
    regexp_replace(col("Raw_Income"), "[$,]", "").cast(DoubleType())
)

print("Sample Income Data:")
income_df.show(5, truncate=False)

# Δημιουργία του View
income_df.createOrReplaceTempView("income_data")

# ΤΕΛΙΚΟ SQL QUERY
final_query = """
WITH Community_Stats AS (
    SELECT 
        c.COMM, 
        SUM(c.POP20) as Total_Pop,
        AVG(i.income_val) as Avg_Income
    FROM census_data c
    JOIN income_data i ON c.ZipCode = i.ZipCode
    GROUP BY c.COMM
),
Crime_Counts AS (
    SELECT 
        COMM, 
        COUNT(*) as Total_Crimes
    FROM crime_data
    GROUP BY COMM
)
SELECT 
    s.COMM,
    s.Avg_Income,
    (c.Total_Crimes / s.Total_Pop) as Crime_Rate_Per_Capita
FROM Community_Stats s
JOIN Crime_Counts c ON s.COMM = c.COMM
WHERE s.Total_Pop > 0 AND s.Avg_Income IS NOT NULL
"""

analysis_df = spark.sql(final_query)

print("Final Analysis Results:")
analysis_df.show(5)

print("\n=== Final Query Execution Plan ===")
analysis_df.explain(mode="extended")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Sample Income Data:
+-------+----------+----------+
|ZipCode|Raw_Income|income_val|
+-------+----------+----------+
|90001  |$52,806   |52806.0   |
|90002  |$46,159   |46159.0   |
|90003  |$47,733   |47733.0   |
|90004  |$54,947   |54947.0   |
|90005  |$44,913   |44913.0   |
+-------+----------+----------+
only showing top 5 rows

Final Analysis Results:
+-------------+------------------+---------------------+
|         COMM|        Avg_Income|Crime_Rate_Per_Capita|
+-------------+------------------+---------------------+
|     Van Nuys| 60215.32467532468|  0.10488271953419896|
|       Encino|         108085.75|  0.08626928626928627|
|Beverly Crest|157253.82558139536|  0.05168652458466186|
|             | 87595.66639387864| 1.989894948082992...|
|  Canoga Park|  71956.1737804878|  0.09899883436856308|
+-------------+------------------+---------------------+
only showing top 5 rows


=== Final Query Execution Plan ===
== Parsed Logical Plan ==
CTE [Community_Stats, Crime_Counts]
:  :- '

In [12]:
# Συσχέτιση (Correlation) για ΟΛΕΣ τις περιοχές
print("Correlation (All Areas):")
analysis_df.select(corr("Avg_Income", "Crime_Rate_Per_Capita")).show()

# Top 10 Περιοχές με Υψηλότερο Εισόδημα
top10_income = analysis_df.orderBy(col("Avg_Income").desc()).limit(10)
print("Correlation (Top 10 Rich Areas):")
top10_income.select(corr("Avg_Income", "Crime_Rate_Per_Capita")).show()

# Bottom 10 Περιοχές με Χαμηλότερο Εισόδημα
bottom10_income = analysis_df.orderBy(col("Avg_Income").asc()).limit(10)
print("Correlation (Bottom 10 Poor Areas):")
bottom10_income.select(corr("Avg_Income", "Crime_Rate_Per_Capita")).show()

global_end_time = time.time()
print(f"Total Execution Time: {global_end_time - global_start_time} seconds")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Correlation (All Areas):
+---------------------------------------+
|corr(Avg_Income, Crime_Rate_Per_Capita)|
+---------------------------------------+
|                    -0.2833188038095936|
+---------------------------------------+

Correlation (Top 10 Rich Areas):
+---------------------------------------+
|corr(Avg_Income, Crime_Rate_Per_Capita)|
+---------------------------------------+
|                    0.02162780394666356|
+---------------------------------------+

Correlation (Bottom 10 Poor Areas):
+---------------------------------------+
|corr(Avg_Income, Crime_Rate_Per_Capita)|
+---------------------------------------+
|                    -0.7086199332400863|
+---------------------------------------+

Total Execution Time: 77.79248070716858 seconds